# RKS f1mo 分解 + 格点偏移 (TPSS0, MGGA)<br>DFT xc 贡献的 f1ao / f1mo 实现 (with grid-shift)


对标 `06-3-decomp_f1mo_tpss0.ipynb`，但在 `vmat_deriv1` (skeleton Fock 导数) 上额外补上**格点偏移 (grid-shift) 贡献**，以恢复 `f1ao` / `f1mo` 对原子求和的平动不变性。

记 `vmat_deriv1 = dVxc/dR` (skeleton, 格点固定)。其 $\sum_A$ 非零 ($\sim 10^{-5}$，MGGA 略大)，源于格点固定而原子移动。引入格点偏移增量 $\Delta = $ grid-motion of $dVxc/dR$，使得

$$\sum_A (\mathrm{vmat\_deriv1}_A + \Delta_A) = 0$$

格点偏移 $\Δ = +(T_1 + T_2)$ 是一阶梯度格点偏移 (见 `11-1`) 对密度矩阵的微分 $d/dD$：

- $T_1[A, t] = \sum_g \frac{dw_g}{dA_t}\, v_{xc,\chi}\, \Xi_\chi$  (格点权重导数 × Vxc 被积函数；"权重部分")
- $T_2[A, t] = \sum_{g \in A} w_g\, \frac{d}{dr_{tg}}(v_{xc,\chi}\, \Xi_\chi)$  (格点坐标随 $A$ 移动；"泛函部分"，即 per-atom $dVxc/dx_t$ 空间导数)

$T_1 + T_2$ 对 $A$ 求和 $\approx 0$ 的 Vxc 矩阵版本；$\Delta = +(T_1+T_2)$ 恰好抵消 `vmat_deriv1` 的 $\sum_A$ ($\sim 10^{-5}$) 至机器精度。

$T_2$ 的解析形式 (避免 FD)：per-atom $dVxc/dx_t$ = `(vmat_ip_A + vmat_ip_A.T)` (ipip-spatial, 对称化) `+ Vxc_Fock(aoA, -fxc·dsum_rho[t], wA)` (fxc-spatial, 其中 `dsum_rho = drho.sum(0) = -∂ρ/∂r` 是空间密度导数)。


In [ ]:
from pyscf import gto, dft, lib
from pyscf.hessian import rks as rks_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from pyscf.grad import rks as rks_grad
from pyscf.dft import numint
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")


In [ ]:
import sys
sys.path.append("..")

from pyhessref.nimatmul.becke_partition import becke_partition
from pyhessref.nimatmul import rks as rks_nimatmul


In [ ]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()


In [ ]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True


In [ ]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
aoslices = mol.aoslice_by_atom()
ni = mf._numint


## 1. Reference f1ao from PySCF


In [ ]:
# PySCF 的 make_h1 返回 f1ao (CPHF/CPKS 的右端)
mf_hess = mf.Hessian()
mf_hess.auxbasis_response = 2
f1ao_ref = mf_hess.make_h1(mo_coeff, mo_occ)
print("f1ao_ref shape:", f1ao_ref.shape)
print("f1ao_ref fp:   ", lib.fp(f1ao_ref))


## 2. f1ao 分解: hcore, J, K, Vxc_deriv1


In [ ]:
# Hybrid 系数
omega, alpha, hyb = ni.rsh_and_hybrid_coeff(mf.xc, spin=mol.spin)
print(f"omega={omega}, hyb={hyb}")

# Vxc_deriv1 (DFT xc 部分, skeleton/格点固定)
vxc_deriv1_ref = rks_hess._get_vxc_deriv1(mf_hess, mo_coeff, mo_occ, 4000)
print("vxc_deriv1_ref fp:", lib.fp(vxc_deriv1_ref))

# J/K 部分 (from DF-RHF _gen_jk)
gen_jk = list(df_rhf_hess._gen_jk(mf_hess, mo_coeff, mo_occ, with_k=True))
h1ao = np.array([r[1] for r in gen_jk])  # hcore derivative
j1ao = np.array([r[2] for r in gen_jk])  # J derivative
k1ao = np.array([r[3] for r in gen_jk])  # K derivative
print("h1ao (hcore) fp:", lib.fp(h1ao))
print("j1ao fp:", lib.fp(j1ao))
print("k1ao fp:", lib.fp(k1ao))


In [ ]:
# 验证分解
f1ao_recap = vxc_deriv1_ref + h1ao + j1ao - 0.5 * hyb * k1ao
print("f1ao decomposition verified:", np.allclose(f1ao_recap, f1ao_ref))
print("max abs diff:", np.max(np.abs(f1ao_recap - f1ao_ref)))


## 3. 格点、AO、rho、vxc、fxc 准备

使用 `grids.build(sort_grids=False)` 以获得 `atm_idx` (格点按原子分组)，这是格点偏移所需的。格点集合与 `nh3_r_tpss0.npz` 中的相同 (仅顺序不同)；由于 f1ao 对格点求和，顺序不影响 `f1ao_recap == f1ao_ref` 的验证。


In [ ]:
grids = dft.gen_grid.Grids(mol)
grids.build(sort_grids=False)
ngrids = grids.weights.size
weights = grids.weights
coords = grids.coords

ao = ni.eval_ao(mol, grids.coords, deriv=2)
print("ao shape:", ao.shape)

rho = ni.eval_rho2(mol, ao[:10], mo_coeff, mo_occ, None, "MGGA")
vxc, fxc = ni.eval_xc_eff(mf.xc, rho, 2, xctype="MGGA")[1:3]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)


## 4. AO 导数指标常量


In [ ]:
# 方向指标
TX, TY, TZ = 0, 1, 2
# AO 导数指标 (deriv=2 共有 10 个分量)
O = 0           # 值
X, Y, Z = 1, 2, 3          # 一阶导
XX, XY, XZ = 4, 5, 6       # 二阶导
YX, YY, YZ = 5, 7, 8       # (YX=XY)
ZX, ZY, ZZ = 6, 8, 9       # (ZX=XZ, ZY=YZ)

GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]


## 5. 自行实现 Vxc_deriv1 (skeleton) + grid-shift (merged)

将 `vmat_deriv1` (skeleton Fock 导数, 格点固定) 与格点偏移增量 $\Delta = T_1 + T_2$ **合并计算**，复用 per-atom 中间量，类似 11-3 中 t8/t9 与 `dao_vxc_diag`/`dao_vxc` 合并：

- **`vmat_ip` (full, for ipip) + per-atom `vmat_ip_A` (for $T_2$'s ipip)**：per-atom 计算 `vmat_ip_A`，累加得 `vmat_ip` (full)。$T_2$'s ipip = `vmat_ip_A + vmat_ip_A.T`。
- **`dR_rho1` (for fxc) + `dsum_rho` (for $T_2$'s fxc)**：同一 per-atom loop 累加 `dsum_rho = $\sum_A$ dR_rho1 = $-\partial\rho/\partial r$`。
- **`T1` (Vxc Fock with `dw`)**：与 `vmat_ip` 同为 Vxc Fock，在同一 loop 计算。

$\Delta = T_1 + T_2$；`vmat_deriv1_grid = vmat_deriv1 + Δ`。`vmat_deriv1` (skeleton) 保留用于验证与 PySCF 参考一致。


In [ ]:
# --- helpers (Vxc-style Fock + per-atom gradient Vxc) ---
def vxc_fock(ao_, veff_, wg_):
    """Vxc-style Fock = sum_g wg * (veff_chi * Xi_chi), with 0.5 factors (nr_vxc convention)."""
    wv_ = wg_ * veff_
    wv_[0] *= 0.5
    wv_[4] *= 0.5
    aow_ = np.einsum("xg, xgu -> gu", wv_[:4], ao_[:4])
    V = aow_.T @ ao_[0]
    V = V + V.T
    for j in range(1, 4):
        aow_t = wv_[4][:, None] * ao_[j]
        V += aow_t.T @ ao_[j]
    return V

def vmat_ip_on_A(aoA, vxcA, wA):
    """per-atom gradient Vxc (vmat_ip on A's grids, non-symmetric, with 0.5 factors)."""
    wv_ = wA * vxcA
    wv_[0] *= 0.5
    wv_[4] *= 0.5
    aow_ = np.einsum("xg, xgu -> gu", wv_[:4], aoA[:4])
    ipA = np.zeros((3, nao, nao))
    for t in range(3):
        ipA[t] += aoA[t + 1].T @ aow_
    aow_d = np.array([wv_[0][:, None] * aoA[d] for d in [X, Y, Z]])
    aow_d[TX] += wv_[1][:, None] * aoA[XX] + wv_[2][:, None] * aoA[XY] + wv_[3][:, None] * aoA[XZ]
    aow_d[TY] += wv_[1][:, None] * aoA[YX] + wv_[2][:, None] * aoA[YY] + wv_[3][:, None] * aoA[YZ]
    aow_d[TZ] += wv_[1][:, None] * aoA[ZX] + wv_[2][:, None] * aoA[ZY] + wv_[3][:, None] * aoA[ZZ]
    for t in range(3):
        ipA[t] += aow_d[t].T @ aoA[O]
    for r in range(3):
        aow_tau = wv_[4][:, None] * aoA[r + 1]
        for t in range(3):
            ipA[t] += aoA[GGA_CALLS[t][r]].T @ aow_tau
    return ipA

# Becke partition dw (grid-weight derivative w.r.t. nuclear coords)
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
adjustment_factor = np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
becke_result = becke_partition(grids.coords, mol.atom_coords(), grids.atm_idx, grids.quadrature_weights, adjustment_factor, 3, 512, 2, None)
dw = becke_result["dw"]


### 5.1 `vmat_ip` (full) + per-atom `vmat_ip_A` ($T_2$'s ipip) + `T1`

`vmat_ip` (full gradient Vxc, for vmat_deriv1's ipip) = $\sum_A$ `vmat_ip_A` (per-atom, for $T_2$'s ipip)。per-atom loop 计算 `vmat_ip_A` 并累加 `vmat_ip`。`T1`[A, t] = Vxc Fock with `dw`[A, t] (格点权重导数部分)，与 `vmat_ip` 同为 Vxc Fock，在同一 loop 内计算。


In [ ]:
# vmat_ip (full) = sum_A vmat_ip_A (per-atom, accumulated); vmat_ip_A stored for T2's ipip.
# T1[A, t] = Vxc Fock with weights dw[A, t] (grid-shift weight part).
vmat_ip = np.zeros((3, nao, nao))            # full gradient Vxc (accumulated)
vmat_ip_A = np.zeros((natm, 3, nao, nao))    # per-atom (for T2's ipip)
T1 = np.zeros((natm, 3, nao, nao))           # grid-shift weight part: Vxc Fock with dw
for A in range(natm):
    mA = grids.atm_idx == A
    vmat_ip_A[A] = vmat_ip_on_A(ao[:, mA, :], vxc[:, mA], grids.weights[mA])
    vmat_ip += vmat_ip_A[A]                  # accumulate full gradient Vxc
    for t in range(3):
        T1[A, t] = vxc_fock(ao, vxc, dw[A, t])

# verify vmat_ip == -pyscf.grad.rks.get_vxc (gradient Vxc matrix)
exc_ref, v_ip_ref = rks_grad.get_vxc(ni, mol, grids, mf.xc, dm0, max_memory=4000)
print("v_ip matches gradient Vxc matrix:", np.allclose(vmat_ip, -v_ip_ref, atol=1e-10))


### 5.2 `vmat_deriv1` (fxc + ipip, skeleton) + grid-shift $T_2$ (merged)

per-atom loop (pass 1) 同时计算：
- `dR_rho1` (一阶密度格点, for vmat_deriv1's fxc) 累加 `dsum_rho` (for $T_2$'s fxc)；
- vmat_deriv1 的 fxc + ipip (skeleton, $-X - X^T$)；
- $T_2$'s ipip = `vmat_ip_A + vmat_ip_A.T`。

$T_2$'s fxc 需要 `dsum_rho` (所有原子累加)，故在 pass 2 (loop 后) 计算，与 `T1` 一并加到 `vmat_deriv1_grid`。


In [ ]:
ao_dm0 = [numint._dot_ao_dm(mol, ao[i], dm0, None, (0, mol.nbas), mol.ao_loc_nr()) for i in range(4)]
wf = weights * fxc

vmat_deriv1 = np.zeros((natm, 3, nao, nao))       # skeleton (for verification)
vmat_deriv1_grid = np.zeros((natm, 3, nao, nao))   # skeleton + grid_shift
dsum_rho = np.zeros((3, 5, ngrids))                # = sum_A dR_rho1 = -d rho / dr (spatial)

# --- Pass 1: per-atom fxc + ipip (skeleton) + accumulate dsum_rho + T2's ipip ---
for A in range(natm):
    dR_rho1 = rks_hess._make_dR_rho1(ao, ao_dm0, A, aoslices, "MGGA")  # [3, 5, ngrids]
    dsum_rho += dR_rho1  # accumulate for T2's fxc

    # vmat_deriv1's fxc (per-atom, all grids)
    wv_f = np.einsum("xyg, txg -> ytg", wf, dR_rho1)
    wv_f[0] *= 0.5   # LDA: 0.5 for symmetrization
    wv_f[4] *= 0.25  # tau: extra 0.5 (0.5 * 0.25 = 0.125 total)
    aow_f = np.einsum("ctg, cgm -> tgm", wv_f[:4], ao[:4])
    for t in range(3):
        vmat_deriv1[A, t] += aow_f[t].T @ ao[O]
    for j in range(1, 4):
        for t in range(3):
            aow_tau = wv_f[4, t][:, None] * ao[j]
            vmat_deriv1[A, t] += aow_tau.T @ ao[j]

    # vmat_deriv1's ipip (vmat_ip on A's bra rows) + antisym (electron -> nuclear)
    _, _, p0, p1 = aoslices[A]
    vmat_deriv1[A, :, p0:p1, :] += vmat_ip[:, p0:p1, :]
    vmat_deriv1[A] = -vmat_deriv1[A] - vmat_deriv1[A].transpose(0, 2, 1)

    # vmat_deriv1_grid = skeleton + T2's ipip (vmat_ip_A + vmat_ip_A.T)
    vmat_deriv1_grid[A] = vmat_deriv1[A]
    for t in range(3):
        vmat_deriv1_grid[A, t] += vmat_ip_A[A, t] + vmat_ip_A[A, t].T

# --- Pass 2: T1 + T2's fxc (uses dsum_rho, per-atom grids) ---
for A in range(natm):
    mA = grids.atm_idx == A
    aoA = ao[:, mA, :]
    wA = grids.weights[mA]
    for t in range(3):
        # dsum_rho here = +d rho/dr (from _make_dR_rho1, opposite sign of _make_drho's -d rho/dr);
        # fxc-spatial = +fxc*(d rho/dr)*Xi, so use +fxc*dsum_rho (sign flipped vs the _make_drho version).
        fxc_dsum_t = np.einsum("xya, ya -> xa", fxc[:, :, mA], dsum_rho[t][:, mA])
        T2_fxc_A = vxc_fock(aoA, fxc_dsum_t, wA)
        vmat_deriv1_grid[A, t] += T1[A, t] + T2_fxc_A


In [ ]:
# verify skeleton vmat_deriv1 == PySCF reference
print("vmat_deriv1 (skeleton) fp:", lib.fp(vmat_deriv1))
print("vxc_deriv1_ref fp:        ", lib.fp(vxc_deriv1_ref))
print("max abs diff:", np.max(np.abs(vmat_deriv1 - vxc_deriv1_ref)))
print("allclose:", np.allclose(vmat_deriv1, vxc_deriv1_ref, atol=1e-8))
print()
# translational invariance: sum_A vmat_deriv1_grid -> 0 (grid-shift cancels skeleton)
print("|vmat_deriv1 (skeleton)|.sum(A) max           :", np.abs(vmat_deriv1.sum(axis=0)).max())
print("|vmat_deriv1_grid (skeleton + grid-shift)|.sum(A) max:", np.abs(vmat_deriv1_grid.sum(axis=0)).max())
print("|T1|.max  :", np.abs(T1).max(), "  |vmat_ip_A|.max:", np.abs(vmat_ip_A).max())


## 7. 组装 f1ao (with grid-shift) 并变换到 f1mo


In [ ]:
# f1ao = h1ao + j1ao - hyb/2 * k1ao + vmat_deriv1 (with grid-shift)
f1ao_grid = h1ao + j1ao - 0.5 * hyb * k1ao + vmat_deriv1_grid
# grid_shift increment = vmat_deriv1_grid - vmat_deriv1 (skeleton)
grid_shift = vmat_deriv1_grid - vmat_deriv1
print("f1ao_grid - grid_shift == f1ao_ref (skeleton):", np.allclose(f1ao_grid - grid_shift, f1ao_ref))
print("max|grid_shift|:", np.abs(grid_shift).max())

# f1mo = C^T @ f1ao @ C_occ
f1mo_grid = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_grid, mocc)
print("f1mo_grid shape:", f1mo_grid.shape)
print("f1mo_grid fp:  ", lib.fp(f1mo_grid))


## 8. 验证：平动不变性

引入格点偏移后，$\sum_A \mathrm{f1ao}_A$ 与 $\sum_A \mathrm{f1mo}_A$ 应非常接近零 (机器精度)。


In [ ]:
# 平动不变性: sum_A f1ao -> 0 (with grid-shift)
f1ao_ref_sum = f1ao_ref.sum(axis=0)
f1ao_grid_sum = f1ao_grid.sum(axis=0)
print("|f1ao_ref (skeleton)|.sum(A) max :", np.abs(f1ao_ref_sum).max())
print("|f1ao_grid (with grid-shift)|.sum(A) max:", np.abs(f1ao_grid_sum).max())
print()
print("np.abs(f1ao_grid.sum(axis=0)).sum() =", np.abs(f1ao_grid.sum(axis=0)).sum())
print()

# f1mo counterpart
f1mo_ref = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_ref, mocc)
print("|f1mo_ref (skeleton)|.sum(A) max :", np.abs(f1mo_ref.sum(axis=0)).max())
print("|f1mo_grid (with grid-shift)|.sum(A) max:", np.abs(f1mo_grid.sum(axis=0)).max())
print("np.abs(f1mo_grid.sum(axis=0)).sum() =", np.abs(f1mo_grid.sum(axis=0)).sum())


## 总结

在 `06-3` 的 `vmat_deriv1` (skeleton Fock 导数) 基础上，补上格点偏移增量 $\Delta = +(T_1 + T_2)$ (一阶梯度格点偏移的 $d/dD$)，恢复 `f1ao` / `f1mo` 的平动不变性 ($\sum_A \sim 10^{-5}$ 降至机器精度 $\sim 10^{-11}$)。

- $T_1 = \sum_g (dw_g/dA_t)\, v_{xc}\, \Xi$ (权重部分，Vxc Fock with weights dw)
- $T_2 = $ per-atom $dVxc/dx_t$ (泛函部分) = `(vmat_ip_A + vmat_ip_A.T)` (ipip-spatial, 对称化) `+ Vxc_Fock(aoA, -fxc·dsum_rho, wA)` (fxc-spatial)
- `dsum_rho = drho.sum(0) = -∂ρ/∂r` (空间密度导数，来自 `_make_drho`)
- $\Delta = +(T_1 + T_2)$; `vmat_deriv1_grid = vmat_deriv1 + Δ`

复用 `06-3` 的 `vmat_ip` / `vmat_deriv1` 结构 (per-atom split + 0.5 factors)，新增 `T1`/`T2` 与之同构。


In [ ]:
# 保存 vmat_ip 与 grid-shift 结果
dat = dict(np.load("nh3_r_tpss0_decomp.npz"))
dat.update({
    "vmat_ip": vmat_ip,
    "vxc_deriv1": vxc_deriv1_ref,
    "vmat_deriv1": vmat_deriv1,
    "vmat_deriv1_mo": np.einsum("up, Atuv, vi -> Atpi", mo_coeff, vmat_deriv1, mocc),
})
np.savez("nh3_r_tpss0_decomp.npz", **dat)
